In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_curve, average_precision_score, roc_auc_score
)

from sklearn.ensemble import RandomForestClassifier

np.random.seed(42)
print("Imports ready")

In [ ]:
N = 10000
rng = np.random.default_rng(42)

transaction_types = ["Send Money", "Bill Payment", "Merchant Payment"]
locations = ["Mumbai", "Delhi", "Bangalore", "Hyderabad", "Chennai", "Kolkata"]
devices = ["Android", "iOS"]

HIGH_RISK_CITIES = {"Delhi", "Kolkata"}

SCORE_TO_PROB = {
    0:(0.002,0.008),1:(0.006,0.015),2:(0.012,0.025),
    3:(0.020,0.040),4:(0.035,0.065),5:(0.060,0.110),
    6:(0.110,0.200),7:(0.200,0.350),8:(0.320,0.500),
    9:(0.450,0.650),10:(0.600,0.820)
}

rows = []

for _ in range(N):
    amount = round(rng.uniform(10,50000),2)
    hour = int(rng.integers(0,24))
    txn = rng.choice(transaction_types)
    loc = rng.choice(locations)
    dev = rng.choice(devices)

    is_night = (hour<6 or hour>22)
    is_late  = (hour<4 or hour>23)
    high_amt = amount>30000
    med_amt  = 15000<amount<=30000
    send     = txn=="Send Money"
    high_city= loc in HIGH_RISK_CITIES
    android  = dev=="Android"

    score=0

    if high_amt: score+=3
    elif med_amt: score+=1

    if is_late: score+=3
    elif is_night: score+=2

    if send: score+=2
    if high_city and high_amt: score+=1
    if android and send: score+=1

    lo,hi = SCORE_TO_PROB[score]
    p = rng.uniform(lo,hi)

    is_fraud = rng.choice([0,1],p=[1-p,p])

    rows.append([amount,txn,loc,hour,dev,is_fraud])

df = pd.DataFrame(rows,columns=[
    "Amount","TransactionType","Location","Hour","Device","IsFraud"
])

df.to_csv("upi_fraud_dataset.csv",index=False)

print("Fraud rate:",df["IsFraud"].mean())
df.head()

In [ ]:
df = pd.read_csv("upi_fraud_dataset.csv")

# Time
df["HourSin"] = np.sin(2*np.pi*df["Hour"]/24)
df["HourCos"] = np.cos(2*np.pi*df["Hour"]/24)

# Amount
df["LogAmount"] = np.log1p(df["Amount"])

df["AmountBucket"] = pd.cut(
    df["Amount"],
    bins=[0,10000,20000,30000,50000],
    labels=[0,1,2,3]
).astype(int)

# Controlled signal (NO leakage)
df["IsNight"] = ((df["Hour"]<6)|(df["Hour"]>22)).astype(int)
df["SendMoney"] = (df["TransactionType"]=="Send Money").astype(int)
df["HighRiskCity"] = df["Location"].isin(["Delhi","Kolkata"]).astype(int)

# Mild interaction
df["AmountNight"] = df["IsNight"] * df["AmountBucket"]

df.head()

In [ ]:
X = df.drop("IsFraud",axis=1)
y = df["IsFraud"]

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,stratify=y,random_state=42
)

print("Train:",len(X_train),"Test:",len(X_test))
print("Fraud rate train:",y_train.mean())

In [ ]:
num_features = [
    "LogAmount","HourSin","HourCos",
    "AmountBucket","AmountNight",
    "IsNight","SendMoney","HighRiskCity"
]

cat_features = ["TransactionType","Location","Device"]

preprocessor = ColumnTransformer([
    ("num",StandardScaler(),num_features),
    ("cat",OneHotEncoder(handle_unknown="ignore"),cat_features)
])

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

print("Processed shape:",X_train_proc.shape)

In [ ]:


rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=10,
    min_samples_leaf=2,
    class_weight="balanced",   # important for fraud
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train_proc, y_train)

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

y_probs = rf.predict_proba(X_test_proc)[:, 1]

pr_auc  = average_precision_score(y_test, y_probs)
roc_auc = roc_auc_score(y_test, y_probs)

print(f"PR-AUC : {pr_auc:.3f}")
print(f"ROC-AUC: {roc_auc:.3f}")

In [ ]:
from sklearn.metrics import precision_recall_curve
import numpy as np

precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs)

beta = 2  # still recall-weighted but not extreme
best_idx = 0
best_f2 = 0

for i, (p, r) in enumerate(zip(precisions[:-1], recalls[:-1])):
    f2 = (1 + beta**2) * (p * r) / (beta**2 * p + r + 1e-9)
    
    # add minimum precision constraint
    if p >= 0.25:   # <-- THIS is the key line
        if f2 > best_f2:
            best_f2 = f2
            best_idx = i

best_threshold = thresholds[best_idx]

print("Chosen Threshold:", round(best_threshold, 3))
print("Precision:", round(precisions[best_idx], 3))
print("Recall:", round(recalls[best_idx], 3))

In [ ]:
y_pred = (y_probs > best_threshold).astype(int)

from sklearn.metrics import classification_report, confusion_matrix

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("=" * 52)
print("FINAL EVALUATION")
print("=" * 52)

print(classification_report(
    y_test, y_pred,
    target_names=["Legit", "Fraud"],
    digits=3
))

print("Confusion Matrix:")
print(cm)

print("\nBreakdown:")
print(f"  TP (Fraud caught) : {tp}")
print(f"  FN (Fraud missed) : {fn}")
print(f"  FP (Legit blocked): {fp}")
print(f"  TN (Legit passed) : {tn}")

print("\nMetrics:")
print("Recall   :", round(tp / (tp + fn), 3))
print("Precision:", round(tp / (tp + fp), 3))